# **Member 2 Research questions**

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

cd_month = pd.read_csv("/content/cd_month.csv")

The two questions are district-level comparisons. Monthly records are therefore aggregated across the full period for each community district.

In [2]:
districts = (
    cd_month.groupby(["community_district", "cd_label", "borough"], as_index=False)
    .agg(
        new_filings=("new_filings", "sum"),
        new_units=("new_units", "sum"),
        ksi=("ksi", "sum"),
        severe_inj=("severe_inj", "sum"),
        crashes=("crashes", "sum"),
        history_known=("history_known", "sum"),
        repeat_location=("repeat_location", "sum")
    )
)

districts["repeat_share"] = (
    districts["repeat_location"] /
    districts["history_known"].replace(0, np.nan)
)

print(f"Community districts included: {len(districts)}")

Community districts included: 59


## Research Question 1

**Do community districts that are adding more new housing have a larger share of their serious crashes at repeat-risk locations?**

For each district, compare the total proposed new housing units with the proportion of crashes at repeat-risk locations.

**repeat_share = repeat_location / history_known**

Districts with fewer than 50 crashes with known location history are excluded to reduce instability in the calculated share. Spearman correlation tests whether districts with more proposed housing units tend to have a higher repeat-risk share.

In [3]:
rq1 = districts[districts["history_known"] >= 50].copy()

rho1, p1 = stats.spearmanr(rq1["new_units"], rq1["repeat_share"])
overall_repeat_share = (
    rq1["repeat_location"].sum() / rq1["history_known"].sum()
)

print(f"Districts included: {len(rq1)}")
print(f"Spearman correlation: {rho1:.3f}")
print(f"P-value: {p1:.4f}")
print(f"Overall repeat-risk share: {overall_repeat_share:.1%}")

Districts included: 59
Spearman correlation: 0.393
P-value: 0.0021
Overall repeat-risk share: 6.9%


In [4]:
fig1 = px.scatter(
    rq1,
    x="new_units",
    y="repeat_share",
    color="borough",
    hover_name="cd_label",
    hover_data=["new_filings", "history_known", "repeat_location"],
    labels={
        "new_units": "Total proposed new housing units",
        "repeat_share": "Share of crashes at repeat-risk locations",
        "borough": "Borough"
    },
    title=f"New housing units and repeat-risk crash share by district (Spearman ρ = {rho1:.2f})"
)

fig1.update_yaxes(tickformat=".0%")
fig1.add_hline(
    y=overall_repeat_share,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"Overall share: {overall_repeat_share:.1%}"
)
fig1.update_layout(template="plotly_white", height=550)
fig1.show()

### RQ1 — Findings

Across **59 community districts**, the relationship between total proposed new housing units and repeat-risk crash share is positive and moderate (Spearman ρ = **0.393**, p = **0.002**). The overall repeat-risk share is approximately **6.9%**.

This indicates that districts with larger proposed housing pipelines tend to have a higher proportion of crashes at locations with a recorded history of repeat serious crashes.

### RQ1 — Limitations

- **Association, not causation:** the result does not show that new housing causes repeat-risk crashes.
- **Proposed units are not completed housing:** the measure does not confirm that housing was built or occupied.
- **District differences:** population, road network, traffic volume, and district size may affect both housing and crash patterns.
- **Small numerators:** repeat-risk crashes may be relatively few in some districts, making their shares less stable.
- **Geographic aggregation:** district-level results cannot identify specific dangerous intersections or streets.

## Research Question 2

**Is new housing being steered toward or away from districts with a record of serious crashes?**

 do districts with more serious crashes early in the period receive more (or less) new housing later?

The months are split into two halves. For each district, killed or seriously injured people (KSI) in the first half are compared with proposed housing units in the second half. Housing pipelines tend to stay where they already were, so the comparison is repeated after controlling for first-half proposed units (partial Spearman correlation). A bootstrap interval shows how uncertain the result is.

In [5]:
months = sorted(cd_month["year_month"].astype(str).unique())
half = len(months) // 2
early_months, late_months = months[:half], months[half:]

def period_totals(selected_months):
    return (
        cd_month[cd_month["year_month"].astype(str).isin(selected_months)]
        .groupby(["community_district", "cd_label", "borough"], as_index=False)
        .agg(new_units=("new_units", "sum"), ksi=("ksi", "sum"))
    )

halves = period_totals(early_months).merge(
    period_totals(late_months),
    on=["community_district", "cd_label", "borough"],
    suffixes=("_early", "_late")
)

def partial_spearman(x, y, control):
    rx, ry, rc = (stats.rankdata(v) for v in (x, y, control))
    res_x = rx - np.polyval(np.polyfit(rc, rx, 1), rc)
    res_y = ry - np.polyval(np.polyfit(rc, ry, 1), rc)
    return stats.pearsonr(res_x, res_y)

rho_simple, p_simple = stats.spearmanr(halves["ksi_early"], halves["new_units_late"])
rho_partial, p_partial = partial_spearman(halves["ksi_early"], halves["new_units_late"], halves["new_units_early"])
rho_units, _ = stats.spearmanr(halves["new_units_early"], halves["new_units_late"])

rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    pick = rng.integers(0, len(halves), len(halves))
    boot.append(partial_spearman(
        halves["ksi_early"].to_numpy()[pick],
        halves["new_units_late"].to_numpy()[pick],
        halves["new_units_early"].to_numpy()[pick]
    )[0])
ci_low, ci_high = np.nanpercentile(boot, [2.5, 97.5])

top_quarter = halves["ksi_early"] >= halves["ksi_early"].quantile(0.75)
ksi_share = halves.loc[top_quarter, "ksi_early"].sum() / halves["ksi_early"].sum()
units_share_early = halves.loc[top_quarter, "new_units_early"].sum() / halves["new_units_early"].sum()
units_share_late = halves.loc[top_quarter, "new_units_late"].sum() / halves["new_units_late"].sum()

print(f"First half: {early_months[0]} to {early_months[-1]} | second half: {late_months[0]} to {late_months[-1]}")
print(f"Districts included: {len(halves)}")
print(f"First-half KSI vs second-half units: rho = {rho_simple:.2f} (p = {p_simple:.3f})")
print(f"After controlling for first-half units: rho = {rho_partial:.2f} (p = {p_partial:.3f}), bootstrap 95% interval [{ci_low:.2f}, {ci_high:.2f}]")
print(f"Units persist between halves: rho = {rho_units:.2f}")
print(f"The {top_quarter.sum()} districts with the most first-half KSI hold {ksi_share:.1%} of KSI, "
      f"{units_share_early:.1%} of first-half units and {units_share_late:.1%} of second-half units")

First half: 2022-01 to 2024-03 | second half: 2024-04 to 2026-06
Districts included: 59
First-half KSI vs second-half units: rho = 0.28 (p = 0.031)
After controlling for first-half units: rho = 0.08 (p = 0.536), bootstrap 95% interval [-0.15, 0.33]
Units persist between halves: rho = 0.70
The 15 districts with the most first-half KSI hold 36.7% of KSI, 38.7% of first-half units and 35.8% of second-half units


In [6]:
fig2 = px.scatter(
    halves,
    x="ksi_early",
    y="new_units_late",
    color="borough",
    size="new_units_early",
    size_max=28,
    hover_name="cd_label",
    hover_data={"ksi_early": ":,.0f", "new_units_early": ":,.0f", "new_units_late": ":,.0f"},
    labels={
        "ksi_early": "KSI in the first half of the period",
        "new_units_late": "Proposed new housing units in the second half",
        "new_units_early": "Proposed units in the first half",
        "borough": "Borough"
    },
    title=f"Past serious crashes and later housing by district (ρ = {rho_simple:.2f}; {rho_partial:.2f} after controlling for earlier housing)"
)

fig2.update_layout(template="plotly_white", height=550, legend_title_text="Borough (bubble size = first-half units)")
fig2.show()

### RQ2 — Findings

Across **59 community districts**, first-half KSI and second-half proposed housing units are positively related when taken alone (Spearman ρ = **0.28**, p = **0.031**). That link mostly reflects that housing pipelines stay where they already were (first-half and second-half units have ρ = **0.70**). After controlling for first-half proposed units, the relationship almost disappears (ρ = **0.08**, p = **0.54**, bootstrap 95% interval from **−0.15** to **0.33**).

The 15 districts with the most KSI in the first half account for **36.7%** of KSI. They held **38.7%** of proposed units in the first half and **35.8%** in the second half, roughly proportional to their share of serious crashes both times.

This indicates that there is no detectable sign that new housing is steered toward or away from districts with a record of serious crashes. Housing appears to follow where it was already being planned.


### RQ2 — Limitations

- **Short period:** each half covers about 27 months, so the test has limited power. The interval is wide, and a moderate effect could be missed.
- **Pipelines persist:** where housing was planned in the first half is a strong predictor of where it is planned in the second half, which makes steering hard to separate from inertia. The partial correlation reduces this problem but does not remove it.
- **Size effect:** large districts tend to have both more KSI and more proposed units. Nothing is adjusted for population, traffic volume or road length.
- **Proposed units are not completed housing,** and housing decisions depend on zoning, land and market conditions that are not in these data.
- **KSI counts only include crashes placed in a district** (about 82% of crashes, about 21% of fatal ones).

## Overall Conclusion

- **RQ1:** districts with more proposed new housing units tend to have a larger share of crashes at repeat-risk locations.
- **RQ2:** there is no sign that new housing is steered toward or away from districts with more serious crashes. Once earlier housing is taken into account, first-half KSI does not predict second-half proposed units, so housing mostly follows where it was already being planned.

Both results are district-level associations among the crashes that could be placed in a district, and neither shows that housing development causes serious crashes. District size and traffic exposure remain important limitations.

## Export files for the website


In [7]:
districts.to_csv("rq_cd_panel.csv", index=False)
halves.to_csv("rq2_halves_panel.csv", index=False)
fig1.write_html("m2_rq1_scatter.html", include_plotlyjs="cdn")
fig2.write_html("m2_rq2_scatter.html", include_plotlyjs="cdn")
print("Saved: rq_cd_panel.csv, rq2_halves_panel.csv, m2_rq1_scatter.html, m2_rq2_scatter.html")

Saved: rq_cd_panel.csv, rq2_halves_panel.csv, m2_rq1_scatter.html, m2_rq2_scatter.html
